# Exploring our data
## Exploring the data to find based on the following:
## 1. The information about the data
## 2. The statistic of the columns
## 3. Columnns Types
## 4. Missing data

In [1]:
import pandas as pd

In [2]:
data_df = pd.read_csv('../data/raw/flight_price_dataset.csv')

In [3]:
data_df.head()

,Airline,Source,Source Name,Destination,Destination Name,Departure Date & Time,Arrival Date & Time,Duration (hrs),Stopovers,Aircraft Type,Class,Booking Source,Base Fare (BDT),Tax & Surcharge (BDT),Total Fare (BDT),Seasonality,Days Before Departure
0,Malaysian Airlines,CXB,Cox's Bazar Airport,CCU,Netaji Subhas Chandra Bose International Airpo...,2025-11-17 06:25:00,2025-11-17 07:38:10,1.219526,Direct,Airbus A320,Economy,Online Website,21131.225021,5169.683753,26300.908775,Regular,10
1,Cathay Pacific,BZL,Barisal Airport,CGP,"Shah Amanat International Airport, Chittagong",2025-03-16 00:17:00,2025-03-16 00:53:31,0.608638,Direct,Airbus A320,First Class,Travel Agency,11605.395471,200.000000,11805.395471,Regular,14
2,British Airways,ZYL,"Osmani International Airport, Sylhet",KUL,Kuala Lumpur International Airport,2025-12-13 12:03:00,2025-12-13 14:44:22,2.689651,1 Stop,Boeing 787,Economy,Travel Agency,39882.499349,11982.374902,51864.874251,Winter Holidays,83
3,Singapore Airlines,RJH,"Shah Makhdum Airport, Rajshahi",DAC,"Hazrat Shahjalal International Airport, Dhaka",2025-05-30 03:21:00,2025-05-30 04:02:09,0.686054,Direct,Airbus A320,Economy,Direct Booking,4435.607340,200.000000,4635.607340,Regular,56
4,British Airways,SPD,Saidpur Airport,YYZ,Toronto Pearson International Airport,2025-04-25 09:14:00,2025-04-25 23:17:20,14.055609,1 Stop,Airbus A350,Business,Direct Booking,59243.806146,14886.570922,74130.377068,Regular,90


In [4]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57000 entries, 0 to 56999
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Airline                57000 non-null  object 
 1   Source                 57000 non-null  object 
 2   Source Name            57000 non-null  object 
 3   Destination            57000 non-null  object 
 4   Destination Name       57000 non-null  object 
 5   Departure Date & Time  57000 non-null  object 
 6   Arrival Date & Time    57000 non-null  object 
 7   Duration (hrs)         57000 non-null  float64
 8   Stopovers              57000 non-null  object 
 9   Aircraft Type          57000 non-null  object 
 10  Class                  57000 non-null  object 
 11  Booking Source         57000 non-null  object 
 12  Base Fare (BDT)        57000 non-null  float64
 13  Tax & Surcharge (BDT)  57000 non-null  float64
 14  Total Fare (BDT)       57000 non-null  float64
 15  Se

In [6]:
class1 = data_df['Class']
class1

0            Economy
1        First Class
2            Economy
3            Economy
4           Business
            ...     
56995       Business
56996    First Class
56997        Economy
56998        Economy
56999       Business
Name: Class, Length: 57000, dtype: object

In [ ]:
data_df.describe()

In [ ]:
# Exploring missing values in the dataset
missing_values = data_df.isnull().sum()
missing_values

In [ ]:
# Exploring categorical features
categorical_cols = data_df.select_dtypes(include=['object']).columns

In [ ]:
categorical_cols

In [10]:
holiday_counts = data_df['Seasonality'].value_counts()
holiday_counts

Seasonality
Regular            44525
Winter Holidays    10930
Hajj                 942
Eid                  603
Name: count, dtype: int64

In [ ]:
# Exploring numeric features
numeric_cols = data_df.select_dtypes(include=['number']).columns
numeric_cols

In [ ]:
# Exploring all columns in the dataset
data_df.columns

In [ ]:
data_df[['Airline']].value_counts()

In [ ]:
data_df[['Seasonality', 'Class']].value_counts()

# Exploring relationship between columns for identify busines logic

In [ ]:
# Grouping Airline based on total number of number total prices 
data_df.groupby('Airline')['Total Fare (BDT)'].sum().sort_values(ascending=False)

In [ ]:
data_df[['Tax & Surcharge (BDT)', 'Class']]

In [ ]:
# creating a new column 'Discount (BDT)' to analyze the discount given on each flight
data_df['Discount (BDT)'] = data_df['Base Fare (BDT)'] - data_df['Total Fare (BDT)']

In [ ]:
#creating new column highest discunt airline
data_df['Highest Discount Airline'] = data_df.groupby('Airline')['Discount (BDT)'].transform('max') == data_df['Discount (BDT)']

In [ ]:
#creating expensive airline column
data_df['Expensive Airline'] = data_df.groupby('Airline')['Total Fare (BDT)'].transform('max') == data_df['Total Fare (BDT)']

Price Sensitivity Category

Why business cares:
Different customers respond differently to price.

In [ ]:
data_df['Price Sensitivity'] = pd.cut(
    data_df['Total Fare (BDT)'],
    bins=[0, 8000, 15000, 30000, float('inf')],
    labels=['Highly Sensitive', 'Sensitive', 'Moderate', 'Low Sensitive']
)

data_df[['Price Sensitivity', 'Total Fare (BDT)']]

Convenience Score (Stopovers + Duration)

Why business cares:
Passengers prefer shorter, direct flights.

In [ ]:
# Convert columns to numeric, replacing errors with NaN
data_df['Stopovers'] = pd.to_numeric(data_df['Stopovers'], errors='coerce').fillna(0)
data_df['Duration (hrs)'] = pd.to_numeric(data_df['Duration (hrs)'], errors='coerce').fillna(0)

data_df['Convenience Score'] = (
    (1 / (1 + data_df['Stopovers'])) * 0.6 +
    (1 / (1 + data_df['Duration (hrs)'])) * 0.4
)

Last-Minute Booking Flag

Why business cares:
Last-minute travelers behave very differently.

In [ ]:
data_df['Last Minute Booking'] = (data_df['Days Before Departure'] <= 3).astype(int)

High Tax Impact Flag

Why business cares:
High taxes reduce customer satisfaction and competitiveness.

In [ ]:
data_df['Tax Impact (%)'] = (
    data_df['Tax & Surcharge (BDT)'] / data_df['Total Fare (BDT)']
) * 100

data_df['High Tax Impact'] = (data_df['Tax Impact (%)'] > 25).astype(int)


Premium Flight Indicator

Why business cares:
Premium flights follow different pricing strategies.

In [ ]:
data_df['Premium Flight'] = (
    (data_df['Class'].isin(['Business', 'First'])) |
    (data_df['Total Fare (BDT)'] > data_df['Total Fare (BDT)'].quantile(0.75))
).astype(int)

In [6]:
transformed_data_df = pd.read_csv('../data/processed/flight_price_dataset_transformed.csv')

In [7]:
transformed_data_df.head()

,airline,source_code,source_name,destination_code,destination_name,departure_time,arrival_time,duration_hours,stopovers,aircraft_type,...,discount_amount,is_highest_discount,is_most_expensive,price_sensitivity,convenience_score,affordability_score,overall_score,has_high_tax,is_premium,is_last_minute
0,Malaysian Airlines,Cxb,Cox'S Bazar Airport,Ccu,Netaji Subhas Chandra Bose International Airpo...,2025-11-17 06:25:00,2025-11-17 07:38:10,1.219526,Direct,Airbus A320,...,0.0,0,0,Moderate,0.780219,0.089466,0.503918,0,0,0
1,Cathay Pacific,Bzl,Barisal Airport,Cgp,"Shah Amanat International Airport, Chittagong",2025-03-16 00:17:00,2025-03-16 00:53:31,0.608638,Direct,Airbus A320,...,0.0,0,0,Sensitive,0.848658,0.096373,0.547744,0,1,0
2,British Airways,Zyl,"Osmani International Airport, Sylhet",Kul,Kuala Lumpur International Airport,2025-12-13 12:03:00,2025-12-13 14:44:22,2.689651,1 Stop,Boeing 787,...,0.0,0,0,Low Sensitive,0.408411,0.084343,0.278784,0,0,0
3,Singapore Airlines,Rjh,"Shah Makhdum Airport, Rajshahi",Dac,"Hazrat Shahjalal International Airport, Dhaka",2025-05-30 03:21:00,2025-05-30 04:02:09,0.686054,Direct,Airbus A320,...,0.0,0,0,Highly Sensitive,0.837240,0.105913,0.544709,0,0,0
4,British Airways,Spd,Saidpur Airport,Yyz,Toronto Pearson International Airport,2025-04-25 09:14:00,2025-04-25 23:17:20,14.055609,1 Stop,Airbus A350,...,0.0,0,0,Low Sensitive,0.326568,0.081876,0.228691,0,1,0


In [8]:
transformed_data_df.columns

Index(['airline', 'source_code', 'source_name', 'destination_code',
       'destination_name', 'departure_time', 'arrival_time', 'duration_hours',
       'stopovers', 'aircraft_type', 'class_type', 'booking_source',
       'base_fare', 'tax_amount', 'total_fare', 'seasonality',
       'days_before_departure', 'discount_amount', 'is_highest_discount',
       'is_most_expensive', 'price_sensitivity', 'convenience_score',
       'affordability_score', 'overall_score', 'has_high_tax', 'is_premium',
       'is_last_minute'],
      dtype='object')